# 05 · Final Load Preparation

**Objective:** Validate the cleaned dataset, select final columns, and export to formats ready for the Tableau dashboard and any downstream database.

In [ ]:
import pandas as pd, numpy as np, sqlite3, os
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')

In [ ]:
# ── Data-quality checks ─────────────────────────────
print('Duplicate rows:', df.duplicated().sum())
print('\nMissing values per column:')
display(df.isnull().sum().rename('missing'))

In [ ]:
# ── Dtype verification ──────────────────────────────
print('Column dtypes:')
display(df.dtypes.rename('dtype').to_frame())

In [ ]:
# ── Select & reorder final columns ──────────────────
FINAL_COLS = [
    'product_title', 'product_category', 'product_rating',
    'total_reviews', 'purchased_last_month',
    'discounted_price', 'original_price', 'discount_percentage',
    'is_best_seller', 'is_sponsored', 'has_coupon',
    'buy_box_availability', 'delivery_date', 'data_collected_at'
]
# keep only cols that exist (safety net)
final_cols = [c for c in FINAL_COLS if c in df.columns]
df_final = df[final_cols].copy()
print(f'Final shape: {df_final.shape}')
df_final.head()

In [ ]:
# ── Export: cleaned CSV for Tableau ──────────────────
OUT_CSV = '../data/amazon_final.csv'
df_final.to_csv(OUT_CSV, index=False)
size_mb = os.path.getsize(OUT_CSV) / 1024 / 1024
print(f'Saved {OUT_CSV}  ({size_mb:.1f} MB)')

In [ ]:
# ── Export: SQLite for local querying ────────────────
DB_PATH = '../data/amazon_products.db'
conn = sqlite3.connect(DB_PATH)
df_final.to_sql('products', conn, if_exists='replace', index=False)
row_count = pd.read_sql('SELECT COUNT(*) AS n FROM products', conn).iloc[0, 0]
conn.close()
print(f'SQLite: {DB_PATH}  →  {row_count:,} rows written')

In [ ]:
# ── Quick verification: read back from SQLite ───────
conn = sqlite3.connect(DB_PATH)
df_check = pd.read_sql('SELECT * FROM products LIMIT 5', conn)
conn.close()
print('Verification — first 5 rows from SQLite:')
display(df_check)